In [1]:
import zipfile

zip_path = "/content/ag-news-classification-dataset.zip"   # change file name
extract_path = "/content/"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipped successfully!")

Unzipped successfully!


In [2]:
# Cell 1: Install dependencies
!apt-get -qq update
!apt-get -qq install -y poppler-utils tesseract-ocr
!pip -q install pdf2image pytesseract pillow scikit-learn pandas numpy joblib

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package poppler-utils.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...


In [14]:
# Cell 2: Paths + dataset loading + validation

import os
import pandas as pd

# =========================
# Update these file paths
# =========================
TRAIN_PATH = "/content/train.csv"
TEST_PATH  = "/content/test.csv"
PDF_PATH   = "/content/sports.pdf"
# =========================

for path in [TRAIN_PATH, TEST_PATH, PDF_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

# Load AG News CSV files
train_df = pd.read_csv(TRAIN_PATH, header=None)
test_df  = pd.read_csv(TEST_PATH, header=None)

# Basic validation
if train_df.shape[1] < 3 or test_df.shape[1] < 3:
    raise ValueError("CSV files must have at least 3 columns: label, title, description")

# Keep only first 3 columns in case extra columns exist
train_df = train_df.iloc[:, :3].copy()
test_df = test_df.iloc[:, :3].copy()

train_df.columns = ["label", "title", "description"]
test_df.columns = ["label", "title", "description"]

# Clean columns
for df in [train_df, test_df]:
    df["label"] = pd.to_numeric(df["label"], errors="coerce")
    df["title"] = df["title"].fillna("").astype(str)
    df["description"] = df["description"].fillna("").astype(str)
    df["text"] = (df["title"].str.strip() + ". " + df["description"].str.strip()).str.strip()
    df["text"] = df["text"].str.replace(r"\s+", " ", regex=True)

# Drop invalid rows
valid_labels = {1, 2, 3, 4}
train_df = train_df[train_df["label"].isin(valid_labels) & (train_df["text"].str.len() > 0)].reset_index(drop=True)
test_df  = test_df[test_df["label"].isin(valid_labels) & (test_df["text"].str.len() > 0)].reset_index(drop=True)

label_map = {
    1: "World",
    2: "Sports",
    3: "Business",
    4: "Sci/Tech"
}

train_df["category"] = train_df["label"].map(label_map)
test_df["category"] = test_df["label"].map(label_map)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("\nTrain category distribution:")
print(train_df["category"].value_counts())
print("\nUsing PDF:", PDF_PATH)

display(train_df.head())

Train shape: (120000, 5)
Test shape : (7600, 5)

Train category distribution:
category
Business    30000
Sci/Tech    30000
Sports      30000
World       30000
Name: count, dtype: int64

Using PDF: /content/sports.pdf


,label,title,description,text,category
0,3.0,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli...",Wall St. Bears Claw Back Into the Black (Reute...,Business
1,3.0,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...,Carlyle Looks Toward Commercial Aerospace (Reu...,Business
2,3.0,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...,Oil and Economy Cloud Stocks' Outlook (Reuters...,Business
3,3.0,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...,Iraq Halts Oil Exports from Main Southern Pipe...,Business
4,3.0,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco...","Oil prices soar to all-time record, posing new...",Business


In [15]:
# Cell 3: Model training + evaluation + save model

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib

X_train = train_df["text"]
y_train = train_df["label"]
X_test = test_df["text"]
y_test = test_df["label"]

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        lowercase=True,
        strip_accents="unicode",
        max_features=40000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        sublinear_tf=True
    )),
    ("clf", LogisticRegression(
        max_iter=1200,
        solver="liblinear",
        random_state=42
    ))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc:.4f}\n")

print("Classification Report:\n")
print(classification_report(
    y_test,
    y_pred,
    target_names=[label_map[i] for i in sorted(label_map.keys())],
    digits=4
))

MODEL_PATH = "/content/ag_news_ocr_classifier.pkl"
joblib.dump(model, MODEL_PATH)
print(f"\nSaved trained model to: {MODEL_PATH}")

# Quick sanity-check prediction
sample_text = "The company reported strong quarterly profits as stock prices surged in early trading."
sample_pred = model.predict([sample_text])[0]
print(f"\nSample prediction: {label_map[sample_pred]}")

Test Accuracy: 0.9200

Classification Report:

              precision    recall  f1-score   support

       World     0.9360    0.9084    0.9220      1900
      Sports     0.9522    0.9847    0.9682      1900
    Business     0.8941    0.8842    0.8891      1900
    Sci/Tech     0.8970    0.9026    0.8998      1900

    accuracy                         0.9200      7600
   macro avg     0.9198    0.9200    0.9198      7600
weighted avg     0.9198    0.9200    0.9198      7600


Saved trained model to: /content/ag_news_ocr_classifier.pkl

Sample prediction: Business


In [16]:
# Cell 4: OCR utilities + helper functions

from pdf2image import convert_from_path
import pytesseract
from PIL import Image, ImageOps, ImageFilter
from collections import defaultdict
import numpy as np

def preprocess_image_for_ocr(img: Image.Image) -> Image.Image:
    """
    Preprocess a PDF page image for better OCR quality.
    """
    img = img.convert("L")                  # grayscale
    img = ImageOps.autocontrast(img)        # better contrast
    img = img.filter(ImageFilter.SHARPEN)   # sharpen
    img = img.point(lambda x: 0 if x < 160 else 255, "1")  # threshold
    return img.convert("L")

def clean_text(text: str) -> str:
    """
    Normalize extracted OCR text.
    """
    if not isinstance(text, str):
        return ""
    text = text.replace("\n", " ").replace("\t", " ")
    text = " ".join(text.split())
    return text.strip()

def compress_pages(page_list):
    """
    Convert [1,2,3,5,7,8] -> '1-3, 5, 7-8'
    """
    if not page_list:
        return ""
    page_list = sorted(set(page_list))
    ranges = []
    start = prev = page_list[0]

    for p in page_list[1:]:
        if p == prev + 1:
            prev = p
        else:
            ranges.append(str(start) if start == prev else f"{start}-{prev}")
            start = prev = p

    ranges.append(str(start) if start == prev else f"{start}-{prev}")
    return ", ".join(ranges)

def predict_text_category(text: str):
    """
    Predict AG News category for input text.
    """
    cleaned = clean_text(text)
    if len(cleaned) == 0:
        return None, 0.0, None

    probs = model.predict_proba([cleaned])[0]
    pred_label = int(model.predict([cleaned])[0])
    pred_category = label_map[pred_label]
    confidence = float(np.max(probs))
    return pred_category, confidence, probs

In [17]:
# Cell 5: OCR + page-wise classification + final summary + save CSV

print("Converting PDF to images...")
try:
    images = convert_from_path(PDF_PATH, dpi=220)
    MAX_PAGES = 3
    images = images[:MAX_PAGES]
except Exception as e:
    raise RuntimeError(f"Failed to convert PDF to images: {e}")

print(f"Total pages found: {len(images)}")

if len(images) == 0:
    raise ValueError("No pages found in the PDF.")

page_results = []
category_to_pages = defaultdict(list)

# OCR configuration
# --oem 3 = default OCR engine
# --psm 6 = assume a block of text
tess_config = r"--oem 3 --psm 6"

print("\nStarting OCR + classification...\n")

for page_num, img in enumerate(images, start=1):
    try:
        processed_img = preprocess_image_for_ocr(img)
        raw_text = pytesseract.image_to_string(processed_img, config=tess_config)
        text = clean_text(raw_text)

        if len(text) < 40:
            result = {
                "page_no": page_num,
                "category": "No sufficient text found",
                "confidence": 0.0,
                "preview": text[:220]
            }
        else:
            category, confidence, _ = predict_text_category(text)
            result = {
                "page_no": page_num,
                "category": category,
                "confidence": confidence,
                "preview": text[:220]
            }
            category_to_pages[category].append(page_num)

        page_results.append(result)

    except Exception as e:
        page_results.append({
            "page_no": page_num,
            "category": f"Processing error: {str(e)}",
            "confidence": 0.0,
            "preview": ""
        })

print("=" * 100)
print("PAGE-WISE PREDICTIONS")
print("=" * 100)

for item in page_results:
    print(f"Page {item['page_no']} -> {item['category']} (confidence={item['confidence']:.4f})")
    print(f"Preview: {item['preview']}")
    print("-" * 100)

print("\n" + "=" * 100)
print("FINAL CATEGORY -> PAGE NUMBERS")
print("=" * 100)

if len(category_to_pages) == 0:
    print("No categories detected. OCR could not extract enough readable text.")
else:
    for category, pages in category_to_pages.items():
        print(f"{category} -> Pages {compress_pages(pages)}")

# Save detailed results
results_df = pd.DataFrame(page_results)
RESULTS_CSV_PATH = "/content/pagewise_news_predictions.csv"
results_df.to_csv(RESULTS_CSV_PATH, index=False)

print(f"\nSaved results CSV to: {RESULTS_CSV_PATH}")
display(results_df)

Converting PDF to images...
Total pages found: 1

Starting OCR + classification...

PAGE-WISE PREDICTIONS
Page 1 -> Sports (confidence=0.9460)
Preview: India defeated Australia by 6 wickets in the T20 match. Virat Kohli scored a brilliant century and led the team to victory.
----------------------------------------------------------------------------------------------------

FINAL CATEGORY -> PAGE NUMBERS
Sports -> Pages 1

Saved results CSV to: /content/pagewise_news_predictions.csv


,page_no,category,confidence,preview
0,1,Sports,0.945995,India defeated Australia by 6 wickets in the T...


In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
